In [1]:
import sys
sys.path.append('..')

from mdx.analytics.core.utils.io_utils import load_json_from_file
from mdx.analytics.core.schema.config import AppConfig
from mdx.analytics.core.utils.schema_util import messages_to_map,nv_frame_to_messages
from mdx.analytics.core.stream.kafka_message_broker import KafkaMessageBroker
from mdx.analytics.core.stream.state.behavior_state_management import StateMgmt
from mdx.analytics.core.stream.source.source_kafka import KafkaSource
from mdx.analytics.core.transform.calibration.calibration import Calibration
from mdx.analytics.core.stream.sink.sink_kafka import SinkKafka
from mdx.analytics.core.utils.crs import CoordinateReferenceSystem

### Prerequisites

You would need to start the Kafka service and get data flowing before testing the Smart City data workflow in below cells.

Below commands are specific to Shuo's dev machine:

- To start the Kafka service:
```
# In WSL2 console
cd MDX/metropolis-iot/standalone-deployment/docker-compose
sudo ./cleanup_all_datalog.sh
docker compose -f foundational/mdx-foundational.yml --profile rtls_test up -d --pull always --build --force-recreate
```

- To get the data flowing:
```
# In Windows cmd console
cd /d D:\work_directory\Shuo\MDX\behavior-analytics
python playback_frames_from_pbjson_smart_city.py --config configs/smart_city_config.json --playback_filepath ../../../../Shuo_data/MDX/traffic.json
```

In [ ]:
valid_config_path = '../../../configs/smart_city_config.json'
# calibration_path = 'configs/calibration_smart_city.json'
calibration_path = '../../../configs/calibration_smart_city_v3.0.json'

config = AppConfig(**load_json_from_file(valid_config_path))
source = KafkaSource(config)
# Instantiate modules
kafka_message_broker = KafkaMessageBroker(config)
calibration = Calibration(config, calibration_path)

behavior_state_manager = StateMgmt(config)
sink = SinkKafka(config, kafka_message_broker)#, data_preprocessor, data_transformer)

crs_config = config.coordinateReferenceSystem
crs = CoordinateReferenceSystem(crs_config)

In [ ]:
# consumer_raw = kafka_message_broker.get_consumer(config.get_kafka_topic("rawTopic", "mdx-raw"), config.kafka.group)
# batch_frames = kafka_message_broker.read_kafka_raw_data(consumer_raw)

batch_frames = source.read_data()
print(len(batch_frames))
if len(batch_frames)>0:
    print(type(batch_frames[0]))
    print(batch_frames[0])

In [ ]:
batch_frames = calibration.filter_frames_by_sensor_id(batch_frames) 
print(len(batch_frames))

In [ ]:
# enhanced_frames = [calibration.transform_frame(frame) for frame in batch_frames]
# print(len(enhanced_frames))
# if len(enhanced_frames)>0:
#     print(type(enhanced_frames[0]))
#     print(enhanced_frames[0])

In [ ]:
batch_messages = [msg for frame in batch_frames for msg in nv_frame_to_messages(frame)]
print(len(batch_messages))
if len(batch_messages)>0:
    print(type(batch_messages[0]))
    print(batch_messages[0])

In [ ]:
updated_messages = [calibration.transform(msg) for msg in batch_messages]
print(len(updated_messages))
if len(updated_messages)>0:
    print(type(updated_messages[0]))
    print(updated_messages[0])

In [ ]:
updated_messages = calibration.filter_messages_by_roi(updated_messages) 
print(len(updated_messages))
if len(updated_messages)>0:
    print(type(updated_messages[0]))
    print(updated_messages[0])

In [ ]:
# calibration._homography_map.get('HWY_20_AND_UNIVERSITY__WB')
# 'HWY_20_AND_UNIVERSITY__WB' in calibration._sensor_map.keys()

In [ ]:
updated_messages_map = messages_to_map(updated_messages)
print(len(updated_messages_map))
print(type(updated_messages_map))
for k, v in updated_messages_map.items():
    print(k)
    print(v[0].object.type)
    break

In [ ]:
behaviors = [behavior_state_manager.update_behavior(k, v, crs) for k, v in updated_messages_map.items()]
print(behaviors[0][0])
if len(behaviors[0][1])>0:
    print('anomaly')
    print(behaviors[0][1])
       

In [ ]:
sink.write(behaviors)

In [ ]:
def direction(bearing: float, mode: int = 2) -> str:
    # mode = 0 is 4-way direction
    # mode = 1 is 8-way direction
    # mode = 2 is 16-way direction
    
    
    

In [ ]:
bearing=20

idx = int(((bearing+11.25)%360) // 22.5)
arr = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]
print(arr[idx])

In [ ]:
bearing=360

idx = int(((bearing+22.5)%360) // 45)
arr = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
print(arr[idx])

In [ ]:
bearing=46

idx = int(((bearing+45)%360) // 90)
arr = ["N", "E", "S", "W"]
print(arr[idx])